# NILE benchmarks

A structural profile is generated for every benchmark dataset by running SPHINX
against the SPARQL endpoint of the repository holding it. For each dataset this
notebook records the time taken to produce the profile, the number of node
shapes and property shapes the profile contains, and the size of the profile on
disk relative to the size of the dataset it describes.

**Structure.** The notebook runs top to bottom in five parts: the execution
environment is recorded first, then a single endpoint is profiled as a check
that the setup works, then every endpoint is profiled repeatedly in a measured
loop, then the repetitions are aggregated and the datasets sized, and finally
the results are tabulated and written to disk.

**Requirements.** The `nile` package must be importable, and every endpoint
listed in the configuration must be running and reachable before the measured
loop is started.

**Outputs.** One profile per dataset is kept in the directory named by
`PROFILE_DIR`, the per-run measurements are written to `profile_runs.csv`, and
the aggregated results to `profile_benchmarks.csv`.

**Methodology.** Profile generation is repeated `REPEATS` times per dataset and
the mean and standard deviation are reported. Timings are wall clock rather than
CPU time, because the work is dominated by queries sent to the endpoints. Each
run constructs a fresh engine, so that no state carries over between runs. A
dataset whose endpoint fails is recorded as a row carrying the error and the
loop continues, so that one unreachable endpoint does not discard the
measurements already collected.

## 1. Execution environment

The date of the last run is recorded because the measurements depend on the
state of the machine and on the contents of the repositories at that moment,
neither of which is captured by the code itself.

In [1]:
import datetime

x = datetime.datetime.now()
print(x) 

2026-09-15 12:17:39.768155


## 2. Profiling the benchmark datasets

### 2.1 Helper functions

`count_shapes` parses a generated profile and returns its node shape count,
property shape count, and total number of triples. Property shapes are gathered
in two ways and combined: those declared explicitly as `sh:PropertyShape`, and
those attached to a node shape through `sh:property`, which are usually blank
nodes. Counting only the former would report zero for most generators. The two
sets are unioned rather than added, so that a named shape which is also
referenced is counted once.

`snapshot_ttl` and `ttl_files_changed` identify which file a given run
produced. A profile's filename is derived from the endpoint URL and is not
straightforward to reconstruct, so the output directory is compared before and
after each run and the files whose modification time or size changed are taken
to be the output of that run. Comparing snapshots is deliberate: an earlier
version selected files by modification time within a tolerance window, which
also selected the previous dataset's profile whenever a run completed quickly,
silently doubling that dataset's shape counts.

In [2]:
# ------------------------------------------------------
# Benchmarking helpers
# ------------------------------------------------------
import contextlib
import io
import time
from pathlib import Path

import pandas as pd
from rdflib import Graph, RDF
from rdflib.namespace import SH


def count_shapes(ttl_path):
    """Count node shapes and property shapes in a SHACL file.

    Property shapes are counted whether they are declared explicitly
    (a sh:PropertyShape) or attached anonymously via sh:property, which is
    how most generators emit them. The two sets are unioned so a named
    shape that is also referenced is not counted twice.
    """
    graph = Graph()
    graph.parse(ttl_path, format="turtle")

    node_shapes = set(graph.subjects(RDF.type, SH.NodeShape))
    property_shapes = set(graph.subjects(RDF.type, SH.PropertyShape))
    property_shapes |= set(graph.objects(None, SH.property))

    return {
        "node_shapes": len(node_shapes),
        "property_shapes": len(property_shapes),
        "triples": len(graph),
    }


def snapshot_ttl(directory):
    """Map every .ttl under `directory` to its (mtime, size)."""
    base = Path(directory)
    if not base.exists():
        return {}
    return {p: (p.stat().st_mtime_ns, p.stat().st_size) for p in base.rglob("*.ttl")}


def ttl_files_changed(directory, before):
    """The .ttl files that appeared or changed since the `before` snapshot.

    Comparing snapshots rather than timestamps matters when a dataset profiles
    quickly: a time-based check with any slack would also pick up the previous
    dataset's file and double its shape counts.
    """
    after = snapshot_ttl(directory)
    return sorted(p for p, stamp in after.items() if before.get(p) != stamp)

### 2.2 Measured run

`endpoints` maps a label to the SPARQL endpoint of each dataset. Labels are
given explicitly rather than derived from the URL, because QLever serves at the
root of a port and would otherwise be labelled by host and port number. Entries
may be commented out to profile a subset.

Every dataset is profiled `REPEATS` times. Between repetitions the output
directory is cleared, since SPHINX appends a disambiguating suffix to a profile
whose filename is already taken rather than overwriting it; without clearing,
the directory would grow by one file per repetition and the file belonging to a
given run would become ambiguous. Because clearing would otherwise destroy the
profiles themselves, each run also copies its output to `PROFILE_DIR` under the
dataset's label, so that exactly one profile per dataset survives the run and is
identifiable by name.

`DISCARD_FIRST` excludes the first repetition of each dataset from the
aggregation. The endpoints cache query results, so the first run is
systematically slower than those that follow; treating it as a warm-up reports
warm-cache timings consistently. To report cold-cache timings instead, set it to
`False` and clear the endpoint caches externally between repetitions.

Two intervals are measured per run: `extraction_s` covers the SPARQL pattern
extraction, and `shacl_s` covers the generation of the profile from the
extracted patterns. `total_s` is their sum and is the figure reported as the
time taken to produce the profile. The intervals are kept separate because they
differ by orders of magnitude, extraction being network-bound and generation
being local serialisation.

In [15]:
# Working directory for generated profiles; cleared before every repetition.
OUTPUT_DIR = "./ShapesTest2"

# Where a copy of each dataset's profile is kept, named after the dataset. The
# working directory is cleared repeatedly, so without this no profile would
# survive the run, and the generated filenames are derived from endpoint URLs
# rather than from dataset labels.
PROFILE_DIR = "./Profiles"

# Number of times each dataset is profiled.
REPEATS = 11

# Exclude the first repetition of each dataset from the aggregation. The
# endpoints cache query results, so the first run is systematically slower than
# the rest; treating it as a warm-up reports warm-cache timings consistently.
# Set to False, and clear the endpoint caches externally, to report cold-cache
# timings instead.
DISCARD_FIRST = True

# Set to True to see SPHINX's own progress output; False keeps the log short
# when looping over many endpoints.
VERBOSE = False

# Label each endpoint explicitly. QLever serves at the root of a port, so a
# label derived from the URL would just be "acb8computer:7023".
endpoints = {
    "lmdb": "http://acb8computer:7019",
    "Care-SM": "http://acb8computer:7020",
    "Berlin": "http://acb8computer:7021",
    "Chebi": "http://acb8computer:7022",
    "dbpedia": "http://acb8computer:7023",
    "drugbank": "http://acb8computer:7024",
    "kegg": "http://acb8computer:7025",
    "nyt": "http://acb8computer:7026",
    "sp2bench": "http://acb8computer:7027",
    "watdiv": "http://acb8computer:7028",
}

In [16]:
# ------------------------------------------------------
# Profile every benchmark dataset, REPEATS times each
# ------------------------------------------------------
import shutil

from nile.sphinx.sphinx import Engine


# Accept a plain list too, falling back to the last URL segment as the label.
if not isinstance(endpoints, dict):
    endpoints = {e.rstrip("/").split("/")[-1]: e for e in endpoints}


def clear_output_dir(directory):
    """Delete the profiles currently in `directory`.

    SPHINX appends a disambiguating suffix to a profile whose filename is
    already taken rather than overwriting it. Without clearing, each repetition
    would therefore leave an additional file behind, the directory would grow
    with every run, and the file belonging to a given repetition would become
    ambiguous.
    """
    base = Path(directory)
    if not base.exists():
        return
    for path in base.glob("*.ttl"):
        path.unlink()


records = []

for dataset, endpoint in endpoints.items():
    print(f"{dataset}")

    for repetition in range(1, REPEATS + 1):
        print(f"  {repetition}/{REPEATS} ... ", end="", flush=True)

        record = {
            "dataset": dataset,
            "endpoint": endpoint,
            "repetition": repetition,
            "status": "ok",
        }

        clear_output_dir(OUTPUT_DIR)
        before = snapshot_ttl(OUTPUT_DIR)

        sink = io.StringIO()
        redirect = (
            contextlib.nullcontext() if VERBOSE else contextlib.redirect_stdout(sink)
        )

        try:
            with redirect:
                # A fresh engine per run, so no state carries over between runs.
                engine = Engine()

                t0 = time.perf_counter()
                rdf_index = engine.extract_patterns([endpoint])
                t1 = time.perf_counter()
                engine.shacl_generator(rdf_index, OUTPUT_DIR)
                t2 = time.perf_counter()
        except Exception as exc:
            record["status"] = f"{type(exc).__name__}: {exc}"
            records.append(record)
            print(f"failed ({type(exc).__name__})")
            continue

        # 4 decimals: SHACL generation is often only a few milliseconds, which
        # rounds away entirely at 2.
        record["extraction_s"] = round(t1 - t0, 4)
        record["shacl_s"] = round(t2 - t1, 4)
        record["total_s"] = round(t2 - t0, 4)

        written = ttl_files_changed(OUTPUT_DIR, before)
        if not written:
            record["status"] = "no SHACL file found"
            records.append(record)
            print(f"{record['total_s']}s, but no .ttl was located")
            continue

        counts = {"node_shapes": 0, "property_shapes": 0, "triples": 0}
        for path in written:
            for key, value in count_shapes(path).items():
                counts[key] += value

        record.update(counts)
        record["shapes_file"] = ", ".join(p.name for p in written)
        record["shapes_bytes"] = sum(p.stat().st_size for p in written)

        # Keep a copy named after the dataset. Later repetitions overwrite it,
        # so the surviving file is the profile of the final repetition.
        kept = []
        Path(PROFILE_DIR).mkdir(parents=True, exist_ok=True)
        for index, path in enumerate(written):
            suffix = "" if len(written) == 1 else f"_{index + 1}"
            target = Path(PROFILE_DIR) / f"{dataset}{suffix}.ttl"
            shutil.copyfile(path, target)
            kept.append(target.name)
        record["profile_file"] = ", ".join(kept)

        records.append(record)
        print(
            f"{record['total_s']}s, "
            f"{record['node_shapes']} node shapes, "
            f"{record['property_shapes']} property shapes "
            f"-> {record['profile_file']}"
        )

runs = pd.DataFrame(records)

lmdb
  1/11 ... 0.1318s, 4 node shapes, 22 property shapes -> lmdb.ttl
  2/11 ... 0.0786s, 4 node shapes, 22 property shapes -> lmdb.ttl
  3/11 ... 0.0952s, 4 node shapes, 22 property shapes -> lmdb.ttl
  4/11 ... 0.0683s, 4 node shapes, 22 property shapes -> lmdb.ttl
  5/11 ... 0.059s, 4 node shapes, 22 property shapes -> lmdb.ttl
  6/11 ... 0.1032s, 4 node shapes, 22 property shapes -> lmdb.ttl
  7/11 ... 0.0957s, 4 node shapes, 22 property shapes -> lmdb.ttl
  8/11 ... 0.0832s, 4 node shapes, 22 property shapes -> lmdb.ttl
  9/11 ... 0.0806s, 4 node shapes, 22 property shapes -> lmdb.ttl
  10/11 ... 0.0988s, 4 node shapes, 22 property shapes -> lmdb.ttl
  11/11 ... 0.085s, 4 node shapes, 22 property shapes -> lmdb.ttl
Care-SM
  1/11 ... 0.3237s, 21 node shapes, 73 property shapes -> Care-SM.ttl
  2/11 ... 0.3002s, 21 node shapes, 73 property shapes -> Care-SM.ttl
  3/11 ... 0.3346s, 21 node shapes, 73 property shapes -> Care-SM.ttl
  4/11 ... 0.2632s, 21 node shapes, 73 property sha

### 2.3 Aggregation across repetitions

The per-run measurements are written to `profile_runs.csv` before anything is
summarised, so that the raw observations remain available independently of the
aggregation applied to them.

Runs that failed are excluded, as is the first repetition of each dataset when
`DISCARD_FIRST` is set. The mean and standard deviation are then computed per
dataset for each measured interval. The median of `total_s` is reported
alongside: where a distribution is skewed by a single slow run, the median is
the more faithful summary, and a large gap between the two is itself a signal
that the repetitions are not homogeneous.

The shape counts and the profile size should be identical on every repetition,
since the same dataset is being profiled each time. `profile_stable` records
whether that held. Where it is `False`, the counts reported for that dataset are
one sample rather than a property of the dataset, and the cause should be
established before the numbers are used.

In [17]:
# ------------------------------------------------------
# Aggregate the repetitions
# ------------------------------------------------------
runs.to_csv("profile_runs.csv", index=False)

MEASURED = ["extraction_s", "shacl_s", "total_s"]

measured = runs[runs["status"] == "ok"].copy()
if DISCARD_FIRST and REPEATS > 1:
    measured = measured[measured["repetition"] > 1]

if measured.empty:
    raise RuntimeError("No successful runs to aggregate; inspect `runs`.")

aggregations = {"repetitions": ("total_s", "size")}
for column in MEASURED:
    aggregations[f"{column}_mean"] = (column, "mean")
    aggregations[f"{column}_std"] = (column, "std")
aggregations["total_s_median"] = ("total_s", "median")
aggregations["node_shapes"] = ("node_shapes", "first")
aggregations["property_shapes"] = ("property_shapes", "first")
aggregations["shapes_bytes"] = ("shapes_bytes", "first")
aggregations["shapes_file"] = ("shapes_file", "last")
aggregations["profile_file"] = ("profile_file", "last")

summary = measured.groupby("dataset").agg(**aggregations).round(4).reset_index()

# The profile should be identical on every repetition. If it is not, the shape
# counts below are one sample rather than a property of the dataset, and the
# cause needs investigating before the numbers are reported.
varying = (
    measured.groupby("dataset")[["node_shapes", "property_shapes", "shapes_bytes"]]
    .nunique()
    .max(axis=1)
)
summary["profile_stable"] = summary["dataset"].map(varying).eq(1)

# Datasets that never produced a usable run still deserve a row, carrying the
# reason rather than being dropped silently.
unmeasured = runs[~runs["dataset"].isin(summary["dataset"])]
if not unmeasured.empty:
    failures = unmeasured.groupby("dataset", as_index=False)["status"].last()
    summary = pd.concat([summary, failures], ignore_index=True)

summary["status"] = summary.get("status", pd.Series(dtype=object)).fillna("ok")
benchmarks = summary

if not benchmarks["profile_stable"].fillna(True).all():
    unstable = benchmarks.loc[~benchmarks["profile_stable"].fillna(True), "dataset"]
    print(f"Warning: profile differed between repetitions for {list(unstable)}")

### 2.4 Dataset sizes

Each endpoint is asked how many triples it holds, so that the size of a profile
can be expressed relative to the size of the dataset it describes. The count is
obtained with `SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }`, sent as a
form-encoded POST that both GraphDB and QLever accept.

Two caveats apply to the resulting figure. The count reflects the triples the
endpoint holds after loading, which may be lower than the number of lines in the
source dump, since triple stores discard duplicates when the data is loaded. The
count also covers the default graph as the endpoint defines it, which for a
dataset loaded into named graphs need not be the whole dataset.

The counts are taken once per dataset, after the measured loop rather than
inside it, so that the time spent counting is not included in any timing and the
endpoints are not disturbed while being measured.

In [18]:
# ------------------------------------------------------
# Dataset size: triples per endpoint
# ------------------------------------------------------
import json
import urllib.parse
import urllib.request

COUNT_QUERY = "SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }"
COUNT_TIMEOUT = 600


def triple_count(endpoint, query=COUNT_QUERY, timeout=COUNT_TIMEOUT):
    """Return the number of triples an endpoint reports for the pattern ?s ?p ?o.

    The query is sent as a form-encoded POST, which both GraphDB and QLever
    accept, so the same call works for every endpoint in the benchmark.
    """
    request = urllib.request.Request(
        endpoint,
        data=urllib.parse.urlencode({"query": query}).encode(),
        headers={"Accept": "application/sparql-results+json"},
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        payload = json.load(response)

    return int(payload["results"]["bindings"][0]["n"]["value"])


sizes = []

for dataset, endpoint in endpoints.items():
    print(f"{dataset} ... ", end="", flush=True)
    try:
        count = triple_count(endpoint)
        print(f"{count:,} triples")
    except Exception as exc:
        count = None
        print(f"failed ({type(exc).__name__}: {exc})")
    sizes.append({"dataset": dataset, "triples": count})

# Drop first, so that re-running this cell does not create triples_x / triples_y.
benchmarks = benchmarks.drop(
    columns=["triples", "bytes_per_triple"], errors="ignore"
).merge(pd.DataFrame(sizes), on="dataset", how="left")

benchmarks["triples"] = pd.to_numeric(benchmarks["triples"], errors="coerce")

if "shapes_bytes" in benchmarks.columns:
    benchmarks["bytes_per_triple"] = (
        benchmarks["shapes_bytes"] / benchmarks["triples"]
    ).round(6)

lmdb ... 227,073 triples
Care-SM ... 73 triples
Berlin ... 40,377 triples
Chebi ... 4,772,706 triples
dbpedia ... 31,704,514 triples
drugbank ... 517,023 triples
kegg ... 1,090,830 triples
nyt ... 335,197 triples
sp2bench ... 50,168 triples
watdiv ... 10,916,457 triples


### 2.5 Results

The table below is the collected result; it is also written to
`profile_benchmarks.csv`, with the individual runs behind it in
`profile_runs.csv`. Its columns are:

| Column | Meaning |
| --- | --- |
| `dataset` | Label of the dataset, as given in `endpoints`. |
| `repetitions` | Successful runs the summary is computed from. |
| `extraction_s_mean` | Mean seconds spent extracting patterns from the endpoint. |
| `shacl_s_mean` | Mean seconds spent generating the profile from those patterns. |
| `total_s_mean`, `total_s_std` | Mean and standard deviation of the time to produce the profile. |
| `total_s_median` | Median of the same, for comparison with the mean. |
| `node_shapes` | Node shapes in the generated profile. |
| `property_shapes` | Property shapes in the generated profile. |
| `triples` | Triples held by the endpoint. |
| `shapes_bytes` | Size of the generated profile on disk, in bytes. |
| `bytes_per_triple` | Profile size relative to dataset size. |
| `profile_file` | Profile kept for this dataset, in `PROFILE_DIR`. |
| `profile_stable` | Whether every repetition produced an identical profile. |
| `status` | `ok`, or the error that prevented the dataset from being profiled. |

A row whose `status` is not `ok` carries no measurements: the dataset either
could not be reached or produced no profile, and it should be excluded from any
figure derived from this table rather than read as a zero.

In [19]:
# ------------------------------------------------------
# Results
# ------------------------------------------------------
columns = [
    "dataset",
    "repetitions",
    "extraction_s_mean",
    "shacl_s_mean",
    "total_s_mean",
    "total_s_std",
    "total_s_median",
    "node_shapes",
    "property_shapes",
    "triples",
    "shapes_bytes",
    "bytes_per_triple",
    "profile_file",
    "profile_stable",
    "status",
]
columns = [c for c in columns if c in benchmarks.columns]

display(benchmarks[columns])

benchmarks.to_csv("profile_benchmarks.csv", index=False)
print("Saved to profile_benchmarks.csv (summary) and profile_runs.csv (per run)")

,dataset,repetitions,extraction_s_mean,shacl_s_mean,total_s_mean,total_s_std,total_s_median,node_shapes,property_shapes,triples,shapes_bytes,bytes_per_triple,profile_file,profile_stable,status
0,Berlin,10,0.3972,0.0007,0.3979,0.0479,0.3842,27,436,40377,61540,1.524135,Berlin.ttl,True,NaN
1,Care-SM,10,0.2746,0.0005,0.2752,0.0302,0.2704,21,73,73,13158,180.246575,Care-SM.ttl,True,NaN
2,Chebi,10,0.0449,0.0005,0.0454,0.0116,0.0426,1,27,4772706,3121,0.000654,Chebi.ttl,True,NaN
3,dbpedia,10,159.3214,0.0988,159.4202,10.5784,157.3148,243,54225,31704514,6652383,0.209824,dbpedia.ttl,True,NaN
4,drugbank,10,0.1030,0.0005,0.1035,0.0098,0.0986,8,215,517023,26147,0.050572,drugbank.ttl,True,NaN
5,kegg,10,0.0500,0.0005,0.0505,0.0070,0.0482,4,49,1090830,4938,0.004527,kegg.ttl,True,NaN
6,lmdb,10,0.0840,0.0007,0.0848,0.0140,0.0841,4,22,227073,2938,0.012939,lmdb.ttl,True,NaN
7,nyt,10,0.0525,0.0003,0.0528,0.0092,0.0502,2,31,335197,3283,0.009794,nyt.ttl,True,NaN
8,sp2bench,10,0.1094,0.0004,0.1098,0.0123,0.1076,7,128,50168,16930,0.337466,sp2bench.ttl,True,NaN
9,watdiv,10,9.2172,0.0010,9.2182,1.6283,9.1196,39,654,10916457,81098,0.007429,watdiv.ttl,True,NaN


Saved to profile_benchmarks.csv (summary) and profile_runs.csv (per run)
